# 06 · MLP en PyTorch sobre datos tabulares, frente a los ensambles del módulo 4

**Módulo 5 · Sesión 13** — Deep learning (puente)

## Objetivos

El notebook 05 construyó la retropropagación a mano. Este la usa como se usa en la
práctica —con PyTorch— sobre los mismos datos, particiones y pliegues de los notebooks
del módulo 4, para responder con números la pregunta de la sesión: **¿cuándo vale la pena
una red neuronal?**

1. Anatomía de un entrenamiento en PyTorch: tensores, `nn.Sequential`, pérdida,
   optimizador, mini-lotes, validación por época y *early stopping*.
2. Qué hacen los **optimizadores** (SGD, momento, Adam) y las tres formas de
   **regularizar** una red (early stopping, *weight decay*, *dropout*), medidas.
3. **Comparación pareada** del MLP contra la regresión logística, Extra-Trees y LightGBM
   sobre Wine Quality, con error estándar y tiempo.
4. Lo mismo sobre **Adult Census** (10× más datos, categóricas de alta cardinalidad).
5. 🔵 Un vistazo a una **red convolucional** sobre los dígitos, para ver el tipo de dato
   en el que las redes sí ganan, y por qué.

La teoría está en `04-redes-neuronales.md` y `05-entrenamiento-y-limites.md`.

**Paquetes:** `numpy`, `pandas`, `matplotlib`, `scikit-learn`, `torch`, `lightgbm`.

In [ ]:
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from lightgbm import LGBMClassifier
from sklearn.compose import ColumnTransformer
from sklearn.datasets import load_digits
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import RepeatedStratifiedKFold, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from torch import nn

warnings.filterwarnings("ignore")
SEMILLA = 42
torch.manual_seed(SEMILLA)
torch.set_num_threads(4)             # CPU; suficiente para estos tamaños
print("PyTorch", torch.__version__, "· GPU disponible:", torch.cuda.is_available(), "(no hace falta)")

## 1. Datos: los mismos del módulo 4

Wine Quality sin duplicados, `buena = quality ≥ 7` (19.7 % positivos), la misma partición
80/20 estratificada y los mismos 5 pliegues de `02-clasificacion-aplicado.ipynb`,
`04-arboles-bagging-aplicado.ipynb` y `06-boosting-aplicado.ipynb`. La métrica es la
**AP**, como allí, y las referencias que hay que superar son las de esos notebooks:
logística 0.52, Random Forest 0.57, Extra-Trees 0.59.

In [ ]:
vinos = pd.read_csv("../datos/wine-quality.csv")
vinos["tipo"] = (vinos["tipo"] == "tinto").astype(int)
vinos = vinos.drop_duplicates().reset_index(drop=True)
vinos["buena"] = (vinos["quality"] >= 7).astype(int)

X = vinos.drop(columns=["quality", "buena"])
y = vinos["buena"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=SEMILLA)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEMILLA)
print(f"Entrenamiento: {len(X_train)} vinos · prueba: {len(X_test)} · {X.shape[1]} variables · "
      f"positivos: {100 * y_train.mean():.1f} %")

## 2. Anatomía de un entrenamiento en PyTorch

Las piezas, una por una, y luego la función que las junta:

- **Tensores**: los datos estandarizados como `float32` (la precisión habitual en redes).
- **Modelo**: `nn.Sequential` apila capas `Linear` (la $\mathbf{XW} + \mathbf{b}$ del
  notebook 05), activaciones `ReLU`, y `Dropout`. La salida es un *logit*; la sigmoide va
  dentro de la pérdida.
- **Pérdida**: `BCEWithLogitsLoss`, la entropía cruzada de la S9 calculada de forma
  numéricamente estable a partir del logit.
- **Optimizador**: recibe los parámetros y aplica la actualización; `Adam` por defecto.
- **Bucle**: por cada época, barajar, recorrer mini-lotes (`zero_grad` → adelante →
  pérdida → `backward` → `step`), y al final evaluar sobre validación en modo `eval`
  (desactiva el dropout). Guardar el mejor estado y **parar** si la validación no mejora
  en `paciencia` épocas.

In [ ]:
def a_tensor(a):
    return torch.tensor(np.asarray(a, dtype=np.float32))


def construir_mlp(p, ocultas=(64, 32), dropout=0.0):
    capas = []
    for h in ocultas:
        capas += [nn.Linear(p, h), nn.ReLU(), nn.Dropout(dropout)]
        p = h
    capas.append(nn.Linear(p, 1))
    return nn.Sequential(*capas)


def entrenar_mlp(X_tr, y_tr, X_va, y_va, ocultas=(64, 32), dropout=0.0, weight_decay=0.0,
                 optimizador="adam", tasa=1e-3, lote=64, epocas=300, paciencia=25, semilla=SEMILLA):
    """Entrena con early stopping sobre (X_va, y_va). Devuelve el modelo con el mejor estado y el historial."""
    torch.manual_seed(semilla)
    X_tr, y_tr, X_va = a_tensor(X_tr), a_tensor(y_tr), a_tensor(X_va)
    y_va = np.asarray(y_va)
    modelo = construir_mlp(X_tr.shape[1], ocultas, dropout)
    perdida = nn.BCEWithLogitsLoss()
    opt = {"adam": lambda: torch.optim.Adam(modelo.parameters(), lr=tasa, weight_decay=weight_decay),
           "sgd": lambda: torch.optim.SGD(modelo.parameters(), lr=tasa, weight_decay=weight_decay),
           "momento": lambda: torch.optim.SGD(modelo.parameters(), lr=tasa, momentum=0.9, weight_decay=weight_decay)}[optimizador]()
    gen = torch.Generator().manual_seed(semilla)
    historial = {"perdida_train": [], "perdida_val": [], "ap_val": []}
    mejor = {"ap": -1.0, "epoca": -1, "estado": None, "sin_mejora": 0}
    for epoca in range(epocas):
        modelo.train()
        for idx in torch.randperm(len(X_tr), generator=gen).split(lote):
            opt.zero_grad()
            perdida(modelo(X_tr[idx]).squeeze(1), y_tr[idx]).backward()
            opt.step()
        modelo.eval()
        with torch.no_grad():
            logit_tr = modelo(X_tr).squeeze(1)
            logit_va = modelo(X_va).squeeze(1)
        p_va = torch.sigmoid(logit_va).numpy()
        historial["perdida_train"].append(perdida(logit_tr, y_tr).item())
        historial["perdida_val"].append(perdida(logit_va, a_tensor(y_va)).item())
        historial["ap_val"].append(average_precision_score(y_va, p_va))
        if historial["ap_val"][-1] > mejor["ap"]:
            mejor.update(ap=historial["ap_val"][-1], epoca=epoca, sin_mejora=0,
                         estado={k: v.clone() for k, v in modelo.state_dict().items()})
        else:
            mejor["sin_mejora"] += 1
            if mejor["sin_mejora"] >= paciencia:
                break
    modelo.load_state_dict(mejor["estado"])
    modelo.eval()
    return modelo, historial, mejor["epoca"]


def predecir(modelo, X):
    with torch.no_grad():
        return torch.sigmoid(modelo(a_tensor(X)).squeeze(1)).numpy()


# Un pliegue, para ver las curvas
idx_tr, idx_va = next(cv.split(X_train, y_train))
escalador = StandardScaler().fit(X_train.iloc[idx_tr])
X_tr, X_va = escalador.transform(X_train.iloc[idx_tr]), escalador.transform(X_train.iloc[idx_va])
y_tr, y_va = y_train.iloc[idx_tr].to_numpy(), y_train.iloc[idx_va].to_numpy()

inicio = time.perf_counter()
modelo, hist, mejor_epoca = entrenar_mlp(X_tr, y_tr, X_va, y_va, epocas=300, paciencia=300)   # sin parar, para ver la curva completa
print(f"300 épocas en {time.perf_counter() - inicio:.1f} s · parámetros: {sum(p.numel() for p in modelo.parameters())} · "
      f"mejor AP de validación {max(hist['ap_val']):.3f} en la época {mejor_epoca}")

fig, ejes = plt.subplots(1, 2, figsize=(13, 4))
ejes[0].plot(hist["perdida_train"], label="entrenamiento")
ejes[0].plot(hist["perdida_val"], label="validación")
ejes[0].axvline(mejor_epoca, color="gray", ls="--", label=f"mejor AP de validación (época {mejor_epoca})")
ejes[0].set_xlabel("época")
ejes[0].set_ylabel("entropía cruzada")
ejes[0].set_title("Pérdida")
ejes[0].legend()
ejes[1].plot(hist["ap_val"], color="C1")
ejes[1].axvline(mejor_epoca, color="gray", ls="--")
ejes[1].set_xlabel("época")
ejes[1].set_ylabel("AP de validación")
ejes[1].set_title("AP de validación por época")
plt.show()

La pérdida de entrenamiento baja sin parar; la de validación toca fondo en las primeras
decenas de épocas y luego sube, mientras la AP de validación se estanca y oscila. Es el
mismo dibujo del notebook 05 y del early stopping de LightGBM (módulo 4, notebook 06). A
partir de aquí, `paciencia=25`: se para 25 épocas después del último máximo de AP y se
recupera el estado de ese máximo.

## 3. Optimizadores y regularización, medidos

### Optimizadores

Tres variantes del descenso del gradiente, sobre el mismo pliegue y la misma tasa:

- **SGD**: $\theta \leftarrow \theta - \eta \nabla \mathcal{L}$, el del módulo 3.
- **Momento**: acumula una media móvil del gradiente, $v \leftarrow 0.9 v + \nabla\mathcal{L}$,
  y se mueve con $v$: acelera en direcciones consistentes y amortigua las oscilaciones.
- **Adam** (Kingma y Ba, 2015): momento **y** una tasa de aprendizaje por parámetro,
  normalizada por la media móvil del gradiente al cuadrado. Es robusto a la tasa
  elegida, y por eso es el valor por defecto en la práctica.

In [ ]:
fig, eje = plt.subplots(figsize=(8, 4.5))
for nombre, cfg in [("SGD (η = 0.01)", dict(optimizador="sgd", tasa=1e-2)),
                    ("SGD + momento (η = 0.01)", dict(optimizador="momento", tasa=1e-2)),
                    ("Adam (η = 0.001)", dict(optimizador="adam", tasa=1e-3))]:
    _, h, _ = entrenar_mlp(X_tr, y_tr, X_va, y_va, epocas=100, paciencia=100, **cfg)
    eje.plot(h["perdida_train"], label=nombre)
    print(f"{nombre:<26} pérdida de entrenamiento tras 100 épocas: {h['perdida_train'][-1]:.3f}   "
          f"mejor AP de validación: {max(h['ap_val']):.3f}")
eje.set_xlabel("época")
eje.set_ylabel("entropía cruzada (entrenamiento)")
eje.set_title("Mismo modelo, mismos datos: el optimizador cambia la velocidad")
eje.legend()
plt.show()

SGD puro con esa tasa se estanca en 0.35; el momento llega en 5 épocas a donde SGD tarda
100, y sigue bajando; Adam va por delante todo el camino (0.25 al final). Con una tasa
mayor SGD también funcionaría — pero habría que buscarla, y Adam ahorra esa búsqueda.
Que la pérdida de **entrenamiento** baje más no significa un modelo mejor: la mejor AP
de validación es la misma para los tres (0.51–0.52); el optimizador cambia cuánto
tarda en llegar, y el early stopping decide dónde parar.

### Regularización

Tres perillas, en CV de 5 pliegues (con early stopping en todos):

- **Early stopping**: parar cuando la validación deja de mejorar. Es regularización
  porque limita cuánto se alejan los pesos de la inicialización.
- ***Weight decay***: la penalización $\ell_2$ de la S7 sobre todos los pesos.
- ***Dropout*** (Srivastava et al., 2014): en cada paso de entrenamiento, apagar al azar
  una fracción de las neuronas de cada capa. Obliga a que ninguna neurona dependa de
  otra concreta; en evaluación se usan todas.

In [ ]:
def evaluar_mlp_cv(cv, X, y, **cfg):
    aps, epocas = [], []
    for idx_tr, idx_va in cv.split(X, y):
        esc = StandardScaler().fit(X.iloc[idx_tr])
        m, _, ep = entrenar_mlp(esc.transform(X.iloc[idx_tr]), y.iloc[idx_tr], esc.transform(X.iloc[idx_va]), y.iloc[idx_va], **cfg)
        aps.append(average_precision_score(y.iloc[idx_va], predecir(m, esc.transform(X.iloc[idx_va]))))
        epocas.append(ep)
    return np.array(aps), np.array(epocas)


filas = []
for nombre, cfg in [("MLP 64-32, sin regularizar (solo early stopping)", {}),
                    ("+ weight decay 1e-3", dict(weight_decay=1e-3)),
                    ("+ dropout 0.2", dict(dropout=0.2)),
                    ("+ dropout 0.2 + weight decay 1e-3", dict(dropout=0.2, weight_decay=1e-3)),
                    ("MLP 256-128, dropout 0.3", dict(ocultas=(256, 128), dropout=0.3)),
                    ("MLP 32, dropout 0.2", dict(ocultas=(32,), dropout=0.2))]:
    inicio = time.perf_counter()
    aps, epocas = evaluar_mlp_cv(cv, X_train, y_train, **cfg)
    filas.append({"configuración": nombre, "AP": aps.mean(), "ee": aps.std(ddof=1) / np.sqrt(len(aps)),
                  "época media de parada": epocas.mean(), "segundos": time.perf_counter() - inicio})
regularizacion = pd.DataFrame(filas).set_index("configuración")
print(regularizacion.round(3).to_string())

Las diferencias entre configuraciones (AP 0.55–0.57) son del orden del error estándar
(≈0.013): sobre 4000 vinos y 12 variables, ni la regularización ni el tamaño de la red
mueven la AP de forma detectable. El early stopping ya hace casi todo el trabajo. Lo que
sí cambia es el **tiempo**: cada configuración cuesta 5–10 s en CV, frente a una fracción
de segundo de la logística. Nos quedamos con dropout 0.2 + weight decay como
configuración razonable (es la habitual en tabular) para la comparación que importa.

## 4. La comparación que importa: MLP contra los modelos del módulo 4

Comparación pareada sobre los mismos 20 pliegues (5 × 4 repeticiones) que
`04-arboles-bagging-aplicado.ipynb` y `06-boosting-aplicado.ipynb`, con el MLP dentro
de un `Pipeline` a través de un envoltorio mínimo, para que reciba el mismo tratamiento.

Un detalle de honestidad: el MLP hace early stopping mirando el pliegue de validación,
que es el mismo sobre el que se calcula su AP. Eso lo favorece (elige la época con la
mejor AP **de ese pliegue**). Para que la comparación sea justa, el envoltorio aparta un
15 % del **entrenamiento** de cada pliegue para el early stopping y no mira el pliegue
de validación.

In [ ]:
class MLPClasificador:
    """Envoltorio con la interfaz de scikit-learn: estandariza, aparta validación interna, entrena con early stopping."""

    def __init__(self, **cfg):
        self.cfg = cfg

    def fit(self, X, y):
        X, y = np.asarray(X), np.asarray(y)
        X_tr, X_va, y_tr, y_va = train_test_split(X, y, test_size=0.15, stratify=y, random_state=SEMILLA)
        self.escalador_ = StandardScaler().fit(X_tr)
        self.modelo_, _, self.epoca_ = entrenar_mlp(self.escalador_.transform(X_tr), y_tr,
                                                    self.escalador_.transform(X_va), y_va, **self.cfg)
        return self

    def predict_proba(self, X):
        p = predecir(self.modelo_, self.escalador_.transform(np.asarray(X)))
        return np.c_[1 - p, p]


modelos = {
    "Regresión logística (S9)": Pipeline([("esc", StandardScaler()), ("clf", LogisticRegression(max_iter=2000))]),
    "Extra-Trees (S10)": ExtraTreesClassifier(n_estimators=300, random_state=SEMILLA, n_jobs=-1),
    "LightGBM por defecto (S11)": LGBMClassifier(random_state=SEMILLA, verbose=-1),
    "MLP 64-32 (dropout 0.2, wd 1e-3)": MLPClasificador(dropout=0.2, weight_decay=1e-3),
}
cv_rep = RepeatedStratifiedKFold(n_splits=5, n_repeats=4, random_state=SEMILLA)
ap_pliegues, tiempos = {}, {}
for nombre, modelo in modelos.items():
    inicio = time.perf_counter()
    aps = []
    for idx_tr, idx_va in cv_rep.split(X_train, y_train):
        modelo.fit(X_train.iloc[idx_tr], y_train.iloc[idx_tr])
        aps.append(average_precision_score(y_train.iloc[idx_va], modelo.predict_proba(X_train.iloc[idx_va])[:, 1]))
    ap_pliegues[nombre] = np.array(aps)
    tiempos[nombre] = (time.perf_counter() - inicio) / len(aps)

resumen = pd.DataFrame({"AP media": {n: a.mean() for n, a in ap_pliegues.items()},
                        "ee": {n: a.std(ddof=1) / np.sqrt(len(a)) for n, a in ap_pliegues.items()},
                        "segundos por pliegue": tiempos})
print(resumen.round(3).to_string())


def comparar(a, b):
    d = ap_pliegues[a] - ap_pliegues[b]
    ee = d.std(ddof=1) / np.sqrt(len(d))
    return pd.Series({"diferencia media": d.mean(), "ee": ee, "cociente": d.mean() / ee}, name=f"{a} − {b}")


print()
print(pd.DataFrame([comparar("MLP 64-32 (dropout 0.2, wd 1e-3)", "Regresión logística (S9)"),
                    comparar("MLP 64-32 (dropout 0.2, wd 1e-3)", "LightGBM por defecto (S11)"),
                    comparar("MLP 64-32 (dropout 0.2, wd 1e-3)", "Extra-Trees (S10)")]).round(3).to_string())

Tres lecturas. Primera, la misma configuración daba AP 0.565 en la sección 3, cuando el
early stopping miraba el pliegue evaluado, y da 0.548 cuando no: el sesgo optimista de
elegir la época sobre los datos de evaluación es de casi dos centésimas — la fuga de
selección del módulo 3, otra vez. Segunda, el MLP gana a la regresión logística (+0.023,
cociente 4.6: aprende interacciones que la logística no tiene) y empata con LightGBM por
defecto. Tercera, queda **por debajo de Extra-Trees por casi cinco centésimas** (cociente
10), y cuesta seis veces más tiempo que Extra-Trees y cien veces más que la logística.
El resultado es el esperable en datos **tabulares pequeños**: los árboles manejan de
fábrica escalas distintas, umbrales y interacciones, mientras que la red tiene que
aprenderlo todo desde pesos aleatorios con 3000 ejemplos. Sobre Wine Quality, la red
neuronal es la peor relación resultado/costo de la tabla.

## 5. Adult Census: diez veces más datos

Quizá el problema es el tamaño. Adult Census (módulo 4, ejercicios) tiene 39 000 filas de
entrenamiento, 14 variables con 8 categóricas de hasta 41 niveles. Para la red, las
categóricas van en *one-hot* (108 columnas); LightGBM las recibe nativas. CV de 5
pliegues, misma partición que el ejercicio 01 del módulo 4.

In [ ]:
adultos = pd.read_csv("../datos/adult-census.csv")
y_ad = adultos["ingreso_alto"]
X_ad = adultos.drop(columns=["ingreso_alto", "particion_original"])
categoricas = X_ad.select_dtypes(exclude="number").columns.tolist()
numericas = [c for c in X_ad.columns if c not in categoricas]
X_ad_train, X_ad_test, y_ad_train, y_ad_test = train_test_split(X_ad, y_ad, test_size=0.2, stratify=y_ad, random_state=SEMILLA)
X_ad_cat = X_ad_train.copy()
for c in categoricas:
    X_ad_cat[c] = X_ad_cat[c].astype("category")

filas = []
ap_adult = {"MLP 128-64": [], "LightGBM (categóricas nativas)": [], "Regresión logística": []}
tiempo_adult = {k: 0.0 for k in ap_adult}
for idx_tr, idx_va in cv.split(X_ad_train, y_ad_train):
    prep = ColumnTransformer([("num", StandardScaler(), numericas),
                              ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categoricas)]).fit(X_ad_train.iloc[idx_tr])
    A, B = prep.transform(X_ad_train.iloc[idx_tr]), prep.transform(X_ad_train.iloc[idx_va])
    y_a, y_b = y_ad_train.iloc[idx_tr].to_numpy(), y_ad_train.iloc[idx_va].to_numpy()

    inicio = time.perf_counter()
    A_tr, A_va, y_a_tr, y_a_va = train_test_split(A, y_a, test_size=0.15, stratify=y_a, random_state=SEMILLA)
    m, _, _ = entrenar_mlp(A_tr, y_a_tr, A_va, y_a_va, ocultas=(128, 64), dropout=0.2, weight_decay=1e-4, lote=256, epocas=60, paciencia=8)
    ap_adult["MLP 128-64"].append(average_precision_score(y_b, predecir(m, B)))
    tiempo_adult["MLP 128-64"] += time.perf_counter() - inicio

    inicio = time.perf_counter()
    lgbm = LGBMClassifier(random_state=SEMILLA, verbose=-1).fit(X_ad_cat.iloc[idx_tr], y_a)
    ap_adult["LightGBM (categóricas nativas)"].append(average_precision_score(y_b, lgbm.predict_proba(X_ad_cat.iloc[idx_va])[:, 1]))
    tiempo_adult["LightGBM (categóricas nativas)"] += time.perf_counter() - inicio

    inicio = time.perf_counter()
    logi = LogisticRegression(max_iter=2000).fit(A, y_a)
    ap_adult["Regresión logística"].append(average_precision_score(y_b, logi.predict_proba(B)[:, 1]))
    tiempo_adult["Regresión logística"] += time.perf_counter() - inicio

resumen_adult = pd.DataFrame({"AP media": {n: np.mean(a) for n, a in ap_adult.items()},
                              "ee": {n: np.std(a, ddof=1) / np.sqrt(len(a)) for n, a in ap_adult.items()},
                              "segundos por pliegue": {n: t / 5 for n, t in tiempo_adult.items()}})
print(resumen_adult.round(3).to_string())
d = np.array(ap_adult["MLP 128-64"]) - np.array(ap_adult["LightGBM (categóricas nativas)"])
print(f"\nMLP − LightGBM: {d.mean():.3f} ± {d.std(ddof=1) / np.sqrt(len(d)):.3f} (cociente {d.mean() / (d.std(ddof=1) / np.sqrt(len(d))):.1f})")

Con diez veces más datos la red mejora en absoluto (AP 0.78), pero LightGBM también
(0.83) y la distancia **se mantiene**: casi cinco centésimas de AP a favor del boosting,
con un cociente que no deja dudas, y otra vez a una fracción del tiempo (0.2 s frente a
2 s por pliegue). El MLP apenas supera a la regresión logística (0.77). No es un resultado
de este dataset: en los *benchmarks* sistemáticos sobre datos tabulares (Grinsztajn et
al., 2022; Shwartz-Ziv y Armon, 2022) los árboles con boosting ganan a las redes en la
mayoría de los conjuntos de tamaño pequeño y mediano, y las redes solo se acercan con
arquitecturas específicas y mucho ajuste.

## 6. 🔵 Donde las redes sí ganan: una red convolucional sobre los dígitos

Las redes neuronales dominan en imágenes, audio y texto por una razón concreta: esos
datos tienen **estructura** que una arquitectura puede explotar y un árbol no. En una
imagen, los píxeles vecinos están relacionados y un trazo es el mismo trazo esté donde
esté. Una **capa convolucional** aplica el mismo filtro pequeño (aquí 3 × 3) en todas las
posiciones: comparte pesos, respeta la localidad, y detecta el mismo patrón en cualquier
lugar. Sobre los dígitos de 8 × 8 (`load_digits`), comparamos regresión logística, MLP y
una CNN mínima, con la accuracy multiclase en CV de 5 pliegues. Y una segunda prueba
que mide el sesgo inductivo directamente: evaluar cada modelo sobre los mismos dígitos
**desplazados un píxel** a la derecha, algo que un ojo humano ni nota.

In [ ]:
digitos = load_digits()
X_dig = (digitos.data / 16.0).astype(np.float32)
y_dig = digitos.target


class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Sequential(nn.Conv2d(1, 16, kernel_size=3, padding=1), nn.ReLU(),
                                  nn.Conv2d(16, 32, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2))
        self.cabeza = nn.Sequential(nn.Flatten(), nn.Linear(32 * 4 * 4, 64), nn.ReLU(), nn.Linear(64, 10))

    def forward(self, x):
        return self.cabeza(self.conv(x.view(-1, 1, 8, 8)))


def entrenar_multiclase(modelo, X_tr, y_tr, epocas=30, lote=64, tasa=1e-3):
    torch.manual_seed(SEMILLA)
    X_tr, y_tr = torch.tensor(X_tr), torch.tensor(y_tr)
    opt = torch.optim.Adam(modelo.parameters(), lr=tasa)
    perdida = nn.CrossEntropyLoss()                      # softmax + entropía cruzada multiclase (S9)
    gen = torch.Generator().manual_seed(SEMILLA)
    for _ in range(epocas):
        modelo.train()
        for idx in torch.randperm(len(X_tr), generator=gen).split(lote):
            opt.zero_grad()
            perdida(modelo(X_tr[idx]), y_tr[idx]).backward()
            opt.step()
    modelo.eval()
    return modelo


def desplazar(X_plano, dx=1):
    """Desplaza cada imagen 8 × 8 dx píxeles a la derecha, rellenando con fondo."""
    im = np.roll(X_plano.reshape(-1, 8, 8), dx, axis=2)
    im[:, :, :dx] = 0
    return im.reshape(-1, 64).astype(np.float32)


filas = []
cv_dig = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEMILLA)
for nombre, fabrica in [("Regresión logística (softmax)", None),
                        ("MLP 64-32", lambda: nn.Sequential(nn.Linear(64, 64), nn.ReLU(), nn.Linear(64, 32), nn.ReLU(), nn.Linear(32, 10))),
                        ("CNN (2 conv + pooling)", CNN)]:
    accs, accs_desplazado, inicio = [], [], time.perf_counter()
    for idx_tr, idx_va in cv_dig.split(X_dig, y_dig):
        if fabrica is None:
            modelo = LogisticRegression(max_iter=3000).fit(X_dig[idx_tr], y_dig[idx_tr])
            predecir_clase = modelo.predict
            n_param = 64 * 10 + 10
        else:
            modelo = entrenar_multiclase(fabrica(), X_dig[idx_tr], y_dig[idx_tr])
            predecir_clase = lambda Z, m=modelo: m(torch.tensor(Z)).argmax(1).numpy()
            n_param = sum(p.numel() for p in modelo.parameters())
        with torch.no_grad():
            accs.append(np.mean(predecir_clase(X_dig[idx_va]) == y_dig[idx_va]))
            accs_desplazado.append(np.mean(predecir_clase(desplazar(X_dig[idx_va])) == y_dig[idx_va]))
    filas.append({"modelo": nombre, "accuracy": np.mean(accs), "ee": np.std(accs, ddof=1) / np.sqrt(5),
                  "accuracy con 1 píxel de desplazamiento": np.mean(accs_desplazado),
                  "parámetros": n_param, "segundos por pliegue": (time.perf_counter() - inicio) / 5})
print(pd.DataFrame(filas).set_index("modelo").round(3).to_string())

Sobre los dígitos tal cual, los tres modelos superan el 96 % —el problema es pequeño y
fácil—, y la CNN es la mejor (2.1 % de error frente a 3.0 % de la logística y 3.6 % del
MLP; el MLP, con diez veces más parámetros que la logística, no la mejora). La segunda
columna es la que enseña: con los dígitos corridos **un píxel**, la logística y el MLP
se desploman a 0.42–0.51 —cada peso está atado a una posición fija— y la CNN resiste
mejor (0.62), porque sus filtros detectan el mismo trazo en cualquier posición. Todos
sufren, porque un píxel es el 12 % de una imagen de 8 × 8 y la cabeza de la CNN sigue
siendo densa; con imágenes reales y aumento de datos la diferencia se vuelve enorme.
El principio es el mismo que hizo ganar a LightGBM en Adult: gana el modelo cuyo **sesgo
inductivo** encaja con la estructura de los datos. Y es el tema del curso de Deep Learning:
convoluciones para imágenes, recurrencia y atención para secuencias, y *transfer
learning* —partir de una red ya entrenada sobre millones de imágenes y ajustar solo la
última capa— para no necesitar millones de ejemplos propios.

In [ ]:
modelo_cnn = entrenar_multiclase(CNN(), X_dig, y_dig)
filtros = modelo_cnn.conv[0].weight.detach().numpy()[:, 0]
fig, ejes = plt.subplots(2, 8, figsize=(12, 3.2))
for eje, f in zip(ejes.ravel(), filtros):
    eje.imshow(f, cmap="RdBu_r", vmin=-np.abs(filtros).max(), vmax=np.abs(filtros).max())
    eje.set_xticks([])
    eje.set_yticks([])
fig.suptitle("Los 16 filtros 3 × 3 de la primera capa convolucional: detectores de bordes y trazos aprendidos")
plt.show()

## 7. Conjunto de prueba

Cerramos como todos los módulos: el mejor modelo de la CV (Extra-Trees) y el MLP,
ajustados sobre todo el entrenamiento y evaluados **una vez** sobre los vinos de prueba.

In [ ]:
filas = []
for nombre in ["Extra-Trees (S10)", "MLP 64-32 (dropout 0.2, wd 1e-3)"]:
    modelo = modelos[nombre].fit(X_train, y_train)
    p = modelo.predict_proba(X_test)[:, 1]
    filas.append({"modelo": nombre, "AP (prueba)": average_precision_score(y_test, p), "AUC-ROC (prueba)": roc_auc_score(y_test, p)})
print(pd.DataFrame(filas).set_index("modelo").round(3).to_string())

## Resumen

| Pregunta | Respuesta medida |
|---|---|
| ¿Qué hay en un entrenamiento? | Tensores, `nn.Sequential`, `BCEWithLogitsLoss`, un optimizador y el bucle de mini-lotes con validación por época y early stopping |
| ¿Optimizador? | Con la misma tasa, SGD se estanca, el momento acelera y Adam va por delante; la mejor AP de validación es la misma: cambia la velocidad, no el destino |
| ¿Regularización? | Dropout y weight decay no mueven la AP de forma detectable sobre 4000 vinos; el early stopping hace el trabajo |
| ¿MLP vs. módulo 4 en Wine? | Gana a la logística (+0.023), empata con LightGBM por defecto, pierde con Extra-Trees (−0.046 ± 0.004), y cuesta 6× más que Extra-Trees y 100× más que la logística |
| ¿Y con 10× más datos (Adult)? | LightGBM 0.83 frente a MLP 0.78 (−0.047 ± 0.001), a una décima parte del tiempo |
| ¿Early stopping sobre el pliegue evaluado? | Infla la AP casi dos centésimas (0.565 frente a 0.548): la validación del early stopping debe salir del entrenamiento |
| ¿Dónde sí? | Datos con estructura que la arquitectura explota: la CNN es la mejor sobre los dígitos y la que menos se desploma cuando se corren un píxel (0.62 frente a 0.42–0.51) |